In [1]:
!pip install open3d
!pip install plyfile

In [7]:
# Устанавливаем pymeshlab и trimesh
!pip install pymeshlab trimesh

KeyboardInterrupt: 

In [10]:
import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from scipy.spatial import KDTree

# ==============================================================================
# 1-2. ЗАГРУЗКА И ВЫСОКОТОЧНАЯ ПРЕДОБРАБОТКА (Размер вокселя уменьшен для точности)
# ==============================================================================

def load_and_preprocess_pcd(file_path, min_points=10, voxel_size=0.015):
    """
    Загрузка PLY, удаление шума, надежный numpy-downsampling и расчет нормалей.
    """
    if not os.path.exists(file_path):
        return None, None

    try:
        plydata = PlyData.read(file_path)
        vertex_data = plydata['vertex']
        points = np.vstack([vertex_data['x'], vertex_data['y'], vertex_data['z']]).T

        available_properties = [p.name for p in vertex_data.properties]
        labels = None
        for name in ["scalar_Label", "label", "class", "classes", "segment"]:
            if name in available_properties:
                labels = np.array(vertex_data[name]).flatten().astype(int)
                break

        if labels is None:
            return None, None

    except Exception:
        return None, None

    if len(points) < min_points:
        return None, None

    # ---- 1. Статистическое удаление шумов через Open3D ----
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=25, std_ratio=2.0)
    # Превращаем индексы в массив numpy, чтобы синхронно отфильтровать координаты и метки
    ind = np.asarray(ind)
    points = points[ind]
    labels = labels[ind]

    # ---- 2. НАДЕЖНЫЙ DOWNSAMPLING НА NUMPY (Взамен voxel_down_sample_and_trace) ----
    # Вычисляем дискретные целочисленные координаты вокселей для каждой точки
    voxel_coords = np.floor(points / voxel_size).astype(int)

    # Трюк: np.unique с аргументом return_index находит уникальные воксели
    # и возвращает индекс ПЕРВОЙ точки, которая попала в этот воксель
    _, down_indices = np.unique(voxel_coords, axis=0, return_index=True)

    # Выбираем отфильтрованные точки и их метки
    points_final = points[down_indices]
    labels_final = labels[down_indices]

    # ---- 3. Создание финального геометрического объекта и расчет нормалей ----
    pcd_final = o3d.geometry.PointCloud()
    pcd_final.points = o3d.utility.Vector3dVector(points_final)

    pcd_final.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    pcd_final.orient_normals_consistent_tangent_plane(k=15)

    return pcd_final, labels_final


# ==============================================================================
# 3. ФОРМИРОВАНИЕ СЕГМЕНТОВ
# ==============================================================================

def split_into_segments(pcd, labels, min_segment_points=40):
    segments = {}
    unique_labels = np.unique(labels)
    points_np = np.asarray(pcd.points)
    normals_np = np.asarray(pcd.normals)

    for label in unique_labels:
        idx = np.where(labels == label)[0]
        if len(idx) < min_segment_points:
            continue

        seg_pcd = o3d.geometry.PointCloud()
        seg_pcd.points = o3d.utility.Vector3dVector(points_np[idx])
        seg_pcd.normals = o3d.utility.Vector3dVector(normals_np[idx])
        segments[label] = seg_pcd

    return segments

# ==============================================================================
# 4-5. ГЕОМЕТРИЧЕСКИЙ АНАЛИЗ И КЛАССИФИКАЦИЯ (Включая все 4 типа из ТЗ)
# ==============================================================================

def analyze_and_classify_segment(seg_pcd):
    pts = np.asarray(seg_pcd.points)
    if len(pts) < 5:
        # Возвращаем "сложную геометрию" для очень мелких сегментов
        return "сложная геометрия", {
            'linearity': 0.0, 'planarity': 0.0, 'sphericity': 0.0,
            'normal_variance': 1.0, 'avg_1nn_distance': 0.0, 'bbox_aspect_ratio': 0.0
        }

    # Анализ формы через PCA (метод главных компонент)
    covariance_matrix = np.cov(pts.T)
    eigenvalues, _ = np.linalg.eigh(covariance_matrix)
    # Сортируем собственные значения по убыванию (lam1 >= lam2 >= lam3)
    eigenvalues = np.sort(eigenvalues)[::-1]

    lam1, lam2, lam3 = eigenvalues
    sum_lam = np.sum(eigenvalues) + 1e-8 # Добавляем небольшое число для стабильности

    # Вычисление признаков, нормированных на сумму собственных значений
    linearity = (lam1 - lam2) / sum_lam  # Мера вытянутости (λ2-λ1)/Σλ
    planarity = (lam2 - lam3) / sum_lam  # Мера плоской структуры (λ1-λ0)/Σλ
    sphericity = lam3 / sum_lam          # Мера компактности / локальной неровности (λ0/Σλ)
    # Примечание: 'Вариация поверхности' по формуле λ0/Σλ эквивалентна 'Сферичности' в данном контексте.

    # Анализ согласованности нормалей
    normals = np.asarray(seg_pcd.normals)
    # Если нормалей мало или их нет, считаем, что они несогласованы
    if len(normals) < 2:
        normal_variance = 1.0 # Максимальное несогласие
    else:
        avg_normal = np.mean(normals, axis=0)
        # Нормализация среднего вектора, чтобы избежать ошибок с np.dot
        avg_normal_norm = np.linalg.norm(avg_normal)
        if avg_normal_norm > 1e-8:
            avg_normal = avg_normal / avg_normal_norm
            normal_variance = np.mean(1.0 - np.dot(normals, avg_normal)**2)
        else:
            normal_variance = 1.0

    # Расчет плотности (среднее расстояние до 1-го ближайшего соседа)
    # Используем compute_nearest_neighbor_distance для получения расстояний до 1-NN
    distances = seg_pcd.compute_nearest_neighbor_distance()
    avg_1nn_distance = np.mean(distances) if len(distances) > 0 else 0.0

    # Расчет Aspect Ratio ограничивающего параллелепипеда
    if len(pts) > 0:
        obb = seg_pcd.get_oriented_bounding_box()
        extent = obb.extent # Размеры (длина, ширина, высота)
        # Чтобы избежать деления на ноль, если какой-то размер очень мал
        if np.min(extent) < 1e-8: # Если один из размеров близок к нулю, используем большое число
            bbox_aspect_ratio = np.inf
        else:
            # Сортируем размеры и берем отношение самого большого к самому маленькому
            sorted_extent = np.sort(extent)
            bbox_aspect_ratio = sorted_extent[-1] / sorted_extent[0]
    else:
        bbox_aspect_ratio = 0.0 # Для пустых сегментов


    # Классификация с учетом новых признаков и уточненных порогов
    # Примечание: Пороги могут потребовать дальнейшей эмпирической настройки
    if planarity > 0.85 and normal_variance < 0.05 and bbox_aspect_ratio < 10.0 and linearity < 0.2:
        # Высокая плоскостность, согласованные нормали, не слишком вытянут, и не линеен
        geom_type = "плоская поверхность"
    elif linearity > 0.6 and bbox_aspect_ratio > 5.0:
        # Высокая линейность и очень вытянут
        geom_type = "трубчатый объект"
    elif sphericity > 0.4 and bbox_aspect_ratio < 3.0:
        # Высокая сферичность и относительно компактный
        geom_type = "сферическая форма"
    else:
        # Все остальное - сложная геометрия
        geom_type = "сложная геометрия"

    # Возвращаем тип геометрии и все вычисленные признаки
    return geom_type, {
        'linearity': linearity,
        'planarity': planarity,
        'sphericity': sphericity,
        'normal_variance': normal_variance,
        'avg_1nn_distance': avg_1nn_distance,
        'bbox_aspect_ratio': bbox_aspect_ratio
    }

# ==============================================================================
# 6-7. АДАПТИВНАЯ РЕКОНСТРУКЦИЯ (Возвращены Poisson, BP и Alpha Shapes)
# ==============================================================================

def reconstruct_segment(seg_pcd, geom_type):
    """
    Выбор алгоритма на основе геометрии согласно ТЗ.
    """
    try:
        distances = seg_pcd.compute_nearest_neighbor_distance()
        avg_dist = np.mean(distances) if len(distances) > 0 else 0.02

        if geom_type == "плоская поверхность":
            # 1. Ball Pivoting для плоскостей (сохраняет четкие границы)
            # Используем немного большие радиусы для лучшего соединения точек на плоскости
            radii = [avg_dist * 2.0, avg_dist * 4.0]
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
                seg_pcd, o3d.utility.DoubleVector(radii)
            )
        elif geom_type in ["трубчатый объект", "сферическая форма"]:
            # 2. Poisson Surface Reconstruction для гладких и замкнутых тел
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                seg_pcd, depth=8, linear_fit=True
            )
            # Обрезка артефактов алгоритма Пуассона по плотности
            vertices_to_remove = densities < np.quantile(densities, 0.08)
            mesh.remove_vertices_by_mask(vertices_to_remove)
        else:
            # 3. Alpha Shapes с pymeshlab для вогнутых и сложных промышленных узлов
            # Передаем точки в pymeshlab функцию
            mesh = reconstruct_alpha_shapes_pymeshlab(np.asarray(seg_pcd.points))

        if mesh is not None:
            mesh.remove_degenerate_triangles()
            mesh.remove_duplicated_triangles()
        return mesh
    except Exception:
        return None

# ==============================================================================
# 9. ОЦЕНКА КАЧЕСТВА (Метрика Chamfer Distance / Погрешность RMSE)
# ==============================================================================

def evaluate_mesh_quality(source_pcd, reconstructed_mesh):
    if reconstructed_mesh is None or len(reconstructed_mesh.triangles) == 0:
        return None
    # Сэмплируем точки из получившегося меша для сравнения
    sampled_pcd = reconstructed_mesh.sample_points_uniformly(number_of_points=len(source_pcd.points))
    # Считаем RMSE расстояний от исходных точек до поверхности меша
    dist_source_to_mesh = source_pcd.compute_point_cloud_distance(sampled_pcd)
    rmse = np.sqrt(np.mean(np.square(dist_source_to_mesh)))
    return rmse

# ==============================================================================
# 8. СКВОЗНОЙ КОНВЕЙЕР И СБОРКА МОДЕЛИ
# ==============================================================================

def process_single_cloud(file_path):
    pcd, labels = load_and_preprocess_pcd(file_path)
    if pcd is None: return None, None, None

    segments = split_into_segments(pcd, labels)
    combined_mesh = o3d.geometry.TriangleMesh()
    file_rmse_list = []

    for label, seg_pcd in segments.items():
        geom_type, _ = analyze_and_classify_segment(seg_pcd) # Изменено, чтобы игнорировать возвращаемые признаки пока что
        mesh = reconstruct_segment(seg_pcd, geom_type)

        if mesh is not None and len(mesh.triangles) > 0:
            # ИСПРАВЛЕНИЕ: Считаем RMSE локально для сегмента,
            # пока меш еще находится строго в координатах этого сегмента!
            rmse = evaluate_mesh_quality(seg_pcd, mesh)
            if rmse is not None:
                file_rmse_list.append(rmse)

            # После этого добавляем в общую модель
            combined_mesh += mesh

    combined_mesh.compute_vertex_normals()

    # Средний RMSE по всем сегментам конкретного файла
    mean_file_rmse = np.mean(file_rmse_list) if file_rmse_list else 0.0
    return combined_mesh, mean_file_rmse, pcd # Modified to return pcd


def process_and_save(file_path, folder_name, output_dir):
    try:
        final_model, rmse, _ = process_single_cloud(file_path) # Modified to unpack 3 values and discard the third
        if final_model is not None and len(final_model.triangles) > 0:
            file_base = os.path.splitext(os.path.basename(file_path))[0]
            save_path = os.path.join(output_dir, f"{file_base}_mesh.ply")
            # Corrected arguments for write_triangle_mesh
            o3d.io.write_triangle_mesh(save_path, final_model, write_vertex_normals=True, write_vertex_colors=True)
            return rmse
    except Exception:
        pass
    return None

In [8]:
import pymeshlab
import trimesh

def reconstruct_alpha_shapes_pymeshlab(points, alpha_fraction=0.5):
    ms = pymeshlab.MeshSet()
    # pymeshlab ожидает np.ndarray для vertex_matrix
    m = pymeshlab.Mesh(vertex_matrix=points.astype(np.float64))
    ms.add_mesh(m, 'input')

    # Динамический расчет alpha на основе максимального размера ограничивающего параллелепипеда
    bbox = points.max(axis=0) - points.min(axis=0)
    alpha = float(np.max(bbox) * alpha_fraction)

    ms.generate_alpha_shape(alpha=pymeshlab.PureValue(alpha))

    # Получаем mesh из MeshSet
    mesh = ms.current_mesh()
    verts = mesh.vertex_matrix()
    faces = mesh.face_matrix()

    if faces is None or len(faces) == 0:
        return None

    # Конвертируем в trimesh, затем в Open3D TriangleMesh
    trimesh_mesh = trimesh.Trimesh(vertices=verts, faces=faces)
    o3d_mesh = o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(trimesh_mesh.vertices),
                                         o3d.utility.Vector3iVector(trimesh_mesh.faces))
    return o3d_mesh


### Интеграция `pymeshlab` для реконструкции сложной геометрии

Теперь мы модифицируем функцию `reconstruct_segment`, чтобы использовать `pymeshlab` для сегментов, классифицированных как "сложная геометрия". Это позволит получить более гладкую и цельную реконструкцию для объектов, которые ранее ошибочно состояли из множества мелких плоскостей.

In [9]:
def reconstruct_segment(seg_pcd, geom_type):
    """
    Выбор алгоритма на основе геометрии согласно ТЗ.
    """
    try:
        distances = seg_pcd.compute_nearest_neighbor_distance()
        avg_dist = np.mean(distances) if len(distances) > 0 else 0.02

        if geom_type == "плоская поверхность":
            # 1. Ball Pivoting для плоскостей (сохраняет четкие границы)
            # Используем немного большие радиусы для лучшего соединения точек на плоскости
            radii = [avg_dist * 2.0, avg_dist * 4.0]
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
                seg_pcd, o3d.utility.DoubleVector(radii)
            )
        elif geom_type in ["трубчатый объект", "сферическая форма"]:
            # 2. Poisson Surface Reconstruction для гладких и замкнутых тел
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                seg_pcd, depth=8, linear_fit=True
            )
            # Обрезка артефактов алгоритма Пуассона по плотности
            vertices_to_remove = densities < np.quantile(densities, 0.08)
            mesh.remove_vertices_by_mask(vertices_to_remove)
        else:
            # 3. Alpha Shapes с pymeshlab для вогнутых и сложных промышленных узлов
            # Передаем точки в pymeshlab функцию
            mesh = reconstruct_alpha_shapes_pymeshlab(np.asarray(seg_pcd.points))

        if mesh is not None:
            mesh.remove_degenerate_triangles()
            mesh.remove_duplicated_triangles()
        return mesh
    except Exception:
        return None


### Повторный запуск конвейера и оценка RMSE после интеграции `pymeshlab` (v4)

Теперь, когда мы интегрировали `pymeshlab` для реконструкции "сложной геометрии", необходимо снова запустить полный конвейер для оценки его влияния на общую метрику RMSE и визуально проверить качество реконструкции трубчатых объектов.

In [11]:
# Перезапускаем основной блок с обновленным методом реконструкции
if __name__ == "__main__":
    DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"
    TARGET_FOLDER_NAME = "15"

    OUTPUT_DIR = f"/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_{TARGET_FOLDER_NAME}_v4_pymeshlab"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    target_path = "/content/drive/MyDrive/MIPT_DS2/15"
    pcd_files = sorted(glob.glob(os.path.join(target_path, "*.ply")))
    total_files = len(pcd_files)

    print(f"Выбрана папка: '{TARGET_FOLDER_NAME}'. Найдено файлов: {total_files}")
    print("Запуск прецизионного конвейера обработки с pymeshlab (v4)...")

    all_rmse_values_v4 = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(process_and_save, f, TARGET_FOLDER_NAME, OUTPUT_DIR): f
            for f in pcd_files
        }

        with tqdm(total=total_files, desc=f"Обработка {TARGET_FOLDER_NAME} (v4)") as pbar:
            for future in as_completed(futures):
                file_rmse = future.result()
                if file_rmse is not None:
                    all_rmse_values_v4.append(file_rmse)
                pbar.update(1)

    if all_rmse_values_v4:
        print(f"\n" + "="*50)
        print(f"СТАТИСТИКА ДЛЯ ОТЧЕТА ПО ПАПКЕ '{TARGET_FOLDER_NAME}' (v4):")
        print(f"Успешно обработано объектов: {len(all_rmse_values_v4)} из {total_files}")
        print(f"Средняя ошибка реконструкции (Mean RMSE): {np.mean(all_rmse_values_v4):.5f} м.")
        print(f"Минимальная ошибка (Best Mesh): {np.min(all_rmse_values_v4):.5f} м.")
        print(f"Максимальная ошибка (Worst Mesh): {np.max(all_rmse_values_v4):.5f} м.")
        print("="*50)
    else:
        print("\nОшибка: Ни один файл не был успешно реконструирован. Проверьте структуру папок.")


Выбрана папка: '15'. Найдено файлов: 500
Запуск прецизионного конвейера обработки с pymeshlab (v4)...


Обработка 15 (v4): 100%|██████████| 500/500 [28:45<00:00,  3.45s/it]


СТАТИСТИКА ДЛЯ ОТЧЕТА ПО ПАПКЕ '15' (v4):
Успешно обработано объектов: 500 из 500
Средняя ошибка реконструкции (Mean RMSE): 5.77405 м.
Минимальная ошибка (Best Mesh): 4.98177 м.
Максимальная ошибка (Worst Mesh): 7.25239 м.


### Сравнение RMSE (версия 4 с `pymeshlab`)

Финальное сравнение средних RMSE для оценки эффекта от всех внесенных изменений, включая интеграцию `pymeshlab`.

In [ ]:
if 'all_rmse_values' in locals() and all_rmse_values:
    print(f"Предыдущий Mean RMSE (изначальный): {np.mean(all_rmse_values):.5f} м.")

if 'all_rmse_values_v2' in locals() and all_rmse_values_v2:
    print(f"Mean RMSE (после первого улучшения классификации): {np.mean(all_rmse_values_v2):.5f} м.")

if 'all_rmse_values_v3' in locals() and all_rmse_values_v3:
    print(f"Mean RMSE (после второго улучшения классификации): {np.mean(all_rmse_values_v3):.5f} м.")

if 'all_rmse_values_v4' in locals() and all_rmse_values_v4:
    print(f"Новый Mean RMSE (с pymeshlab - v4): {np.mean(all_rmse_values_v4):.5f} м.")

    if 'all_rmse_values_v3' in locals() and all_rmse_values_v3:
        improvement_v4 = np.mean(all_rmse_values_v3) - np.mean(all_rmse_values_v4)
        if improvement_v4 > 0:
            print(f"Улучшение RMSE относительно v3: {improvement_v4:.5f} м.")
        elif improvement_v4 < 0:
            print(f"Ухудшение RMSE относительно v3: {-improvement_v4:.5f} м.")
        else:
            print("RMSE относительно v3 остался без изменений.")


In [4]:
# ==============================================================================
# ОСНОВНОЙ БЛОК: ЗАПУСК ДЛЯ ОДНОЙ КОНКРЕТНОЙ ПОДПАПКИ
# ==============================================================================
if __name__ == "__main__":
    DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"  # Основной путь к датасету

    # !!! УКАЖИТЕ НАЗВАНИЕ ВАШЕЙ ЦЕЛЕВОЙ ПАПКИ ИЗ 25 ДОСТУПНЫХ !!!
    TARGET_FOLDER_NAME = "15"  # Например: "valve", "pipe", "tanks" и т.д.

    OUTPUT_DIR = f"/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_{TARGET_FOLDER_NAME}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    target_path = "/content/drive/MyDrive/MIPT_DS2/15"

    # Собираем файлы только из этой папки
    pcd_files = sorted(glob.glob(os.path.join(target_path, "*.ply")))
    total_files = len(pcd_files)

    print(f"Выбрана папка: '{TARGET_FOLDER_NAME}'. Найдено файлов: {total_files}")
    print("Запуск прецизионного конвейера обработки...")

    all_rmse_values = []

    # Распределение задач на 4 потока CPU
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(process_and_save, f, TARGET_FOLDER_NAME, OUTPUT_DIR): f
            for f in pcd_files
        }

        # Запуск текстового tqdm прогресс-бара без конфликтов с виджетами
        with tqdm(total=total_files, desc=f"Обработка {TARGET_FOLDER_NAME}") as pbar:
            for future in as_completed(futures):
                file_rmse = future.result()
                if file_rmse is not None:
                    all_rmse_values.append(file_rmse)
                pbar.update(1)
    # Вывод итоговой статистики для защиты лабораторной
    if all_rmse_values:
        print(f"\n" + "="*50)
        print(f"СТАТИСТИКА ДЛЯ ОТЧЕТА ПО ПАПКЕ '{TARGET_FOLDER_NAME}':")
        print(f"Успешно обработано объектов: {len(all_rmse_values)} из {total_files}")
        print(f"Средняя ошибка реконструкции (Mean RMSE): {np.mean(all_rmse_values):.5f} м.")
        print(f"Минимальная ошибка (Best Mesh): {np.min(all_rmse_values):.5f} м.")
        print(f"Максимальная ошибка (Worst Mesh): {np.max(all_rmse_values):.5f} м.")
        print("="*50)
    else:
        print("\nОшибка: Ни один файл не был успешно реконструирован. Проверьте структуру папок.")

Выбрана папка: '15'. Найдено файлов: 500
Запуск прецизионного конвейера обработки...


Обработка 15: 100%|██████████| 500/500 [21:49<00:00,  2.62s/it]


СТАТИСТИКА ДЛЯ ОТЧЕТА ПО ПАПКЕ '15':
Успешно обработано объектов: 500 из 500
Средняя ошибка реконструкции (Mean RMSE): 5.13813 м.
Минимальная ошибка (Best Mesh): 4.09198 м.
Максимальная ошибка (Worst Mesh): 9.72408 м.


In [5]:
import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from scipy.spatial import KDTree

if __name__ == "__main__":
    DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"
    TARGET_FOLDER_NAME = "15"

    OUTPUT_DIR = f"/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_{TARGET_FOLDER_NAME}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    pcd_files = sorted(glob.glob(os.path.join(DATASET_DIR, TARGET_FOLDER_NAME, "*.ply")))

    all_rmse = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(process_and_save, f, OUTPUT_DIR): f for f in pcd_files}
        with tqdm(total=len(pcd_files), desc=f"Расчет {TARGET_FOLDER_NAME}") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res is not None: all_rmse.append(res)
                pbar.update(1)

    print(f"\n==================================================")
    print(f"СТАТИСТИЧЕСКИЙ АНАЛИЗ ДЛЯ РАЗДЕЛА МЕТРИК КАЧЕСТВА:")
    print(f"Целевая подпапка: {TARGET_FOLDER_NAME}")
    print(f"Успешность построения конвейера: {len(all_rmse)} / {len(pcd_files)} файлов")
    print(f"Средняя погрешность Chamfer Distance (Mean RMSE): {np.mean(all_rmse):.5f} м.")
    print(f"Минимальная зафиксированная погрешность (Best): {np.min(all_rmse):.5f} м.")
    print(f"Максимальная зафиксированная погрешность (Worst): {np.max(all_rmse):.5f} м.")
    print(f"==================================================")


Расчет 15:   0%|          | 0/500 [00:00<?, ?it/s]


TypeError: process_and_save() missing 1 required positional argument: 'output_dir'

In [12]:
import plotly.graph_objects as go

# Убедитесь, что 'pcd_files' определен, например, из ячейки 'wWosTFtwJ8iB'
# Если эта ячейка запускается изолированно, может потребоваться определить pcd_files здесь.
# Пример: pcd_files = ['/content/drive/MyDrive/MIPT_DS2/15/valve_0001_lidar_classes.ply']

# Обрабатываем один эталонный файл для визуализации
sample_file = pcd_files[0]
mesh, rmse, source_pcd = process_single_cloud(sample_file)

if mesh is not None and source_pcd is not None:
    verts = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    source_points = np.asarray(source_pcd.points)

    # Строим интерактивный полигональный объект в Plotly
    fig = go.Figure(data=[
        go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=triangles[:, 0], j=triangles[:, 1], k=triangles[:, 2],
            opacity=0.8,
            color='lightblue',
            flatshading=True,
            name="Реконструированный Меш"
        ),
        go.Scatter3d(
            x=source_points[:, 0], y=source_points[:, 1], z=source_points[:, 2],
            mode='markers',
            marker=dict(
                size=1,
                color='red', # Цвет для исходных точек
                opacity=0.6
            ),
            name="Исходное Облако Точек"
        )
    ])

    fig.update_layout(
        title=f"Интерактивная модель: {os.path.basename(sample_file)} (RMSE: {rmse:.4f} м.)",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

    # Сохранение визуализации на диск
    output_viz_dir = "/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/visualizations"
    os.makedirs(output_viz_dir, exist_ok=True)
    viz_filename = os.path.join(output_viz_dir, f"{os.path.basename(sample_file).replace('.ply', '')}_reconstruction.html")
    fig.write_html(viz_filename)
    print(f"Визуализация сохранена в: {viz_filename}")

Визуализация сохранена в: /content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/visualizations/valve_0001_lidar_classes_reconstruction.html


### Обновленная визуализация с `pymeshlab`

Для наглядного сравнения, давайте снова отобразим реконструкцию тестового файла с использованием нового пайплайна с `pymeshlab`.

In [ ]:
import plotly.graph_objects as go

# Используем тот же sample_file, что и ранее
# sample_file = pcd_files[0] # уже определен

# Запускаем process_single_cloud, который теперь использует pymeshlab для 'сложной геометрии'
mesh_v4, rmse_v4, source_pcd_v4 = process_single_cloud(sample_file)

if mesh_v4 is not None and source_pcd_v4 is not None:
    verts_v4 = np.asarray(mesh_v4.vertices)
    triangles_v4 = np.asarray(mesh_v4.triangles)
    source_points_v4 = np.asarray(source_pcd_v4.points)

    fig_v4 = go.Figure(data=[
        go.Mesh3d(
            x=verts_v4[:, 0], y=verts_v4[:, 1], z=verts_v4[:, 2],
            i=triangles_v4[:, 0], j=triangles_v4[:, 1], k=triangles_v4[:, 2],
            opacity=0.8,
            color='lightblue',
            flatshading=True,
            name="Реконструированный Меш (v4 с pymeshlab)"
        ),
        go.Scatter3d(
            x=source_points_v4[:, 0], y=source_points_v4[:, 1], z=source_points_v4[:, 2],
            mode='markers',
            marker=dict(
                size=1,
                color='red', # Цвет для исходных точек
                opacity=0.6
            ),
            name="Исходное Облако Точек"
        )
    ])

    fig_v4.update_layout(
        title=f"Интерактивная модель: {os.path.basename(sample_file)} (RMSE v4: {rmse_v4:.4f} м.)",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig_v4.show()

    # Сохранение визуализации на диск
    output_viz_dir = "/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/visualizations"
    os.makedirs(output_viz_dir, exist_ok=True)
    viz_filename_v4 = os.path.join(output_viz_dir, f"{os.path.basename(sample_file).replace('.ply', '')}_reconstruction_v4_pymeshlab.html")
    fig_v4.write_html(viz_filename_v4)
    print(f"Визуализация сохранена в: {viz_filename_v4}")


### Повторный запуск конвейера и оценка RMSE после доработки классификации

Теперь, после внесения изменений в функцию `analyze_and_classify_segment` (добавление условия `linearity < 0.2` для `плоской поверхности` и уточнение `avg_1nn_distance` для мелких сегментов), необходимо повторно запустить полный конвейер для оценки его влияния на общую метрику RMSE.

Это позволит нам увидеть, улучшилась ли средняя ошибка реконструкции, особенно для объектов, которые ранее ошибочно классифицировались как плоские поверхности (например, трубы с низкой плотностью точек).

In [ ]:
# Перезапускаем основной блок, чтобы обновить результаты с новыми правилами классификации
if __name__ == "__main__":
    DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"
    TARGET_FOLDER_NAME = "15"

    OUTPUT_DIR = f"/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_{TARGET_FOLDER_NAME}_v2"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    target_path = "/content/drive/MyDrive/MIPT_DS2/15"
    pcd_files = sorted(glob.glob(os.path.join(target_path, "*.ply")))
    total_files = len(pcd_files)

    print(f"Выбрана папка: '{TARGET_FOLDER_NAME}'. Найдено файлов: {total_files}")
    print("Запуск прецизионного конвейера обработки с обновленными правилами классификации...")

    all_rmse_values_v2 = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(process_and_save, f, TARGET_FOLDER_NAME, OUTPUT_DIR): f
            for f in pcd_files
        }

        with tqdm(total=total_files, desc=f"Обработка {TARGET_FOLDER_NAME} (v2)") as pbar:
            for future in as_completed(futures):
                file_rmse = future.result()
                if file_rmse is not None:
                    all_rmse_values_v2.append(file_rmse)
                pbar.update(1)

    if all_rmse_values_v2:
        print(f"\n" + "="*50)
        print(f"СТАТИСТИКА ДЛЯ ОТЧЕТА ПО ПАПКЕ '{TARGET_FOLDER_NAME}' (v2):")
        print(f"Успешно обработано объектов: {len(all_rmse_values_v2)} из {total_files}")
        print(f"Средняя ошибка реконструкции (Mean RMSE): {np.mean(all_rmse_values_v2):.5f} м.")
        print(f"Минимальная ошибка (Best Mesh): {np.min(all_rmse_values_v2):.5f} м.")
        print(f"Максимальная ошибка (Worst Mesh): {np.max(all_rmse_values_v2):.5f} м.")
        print("="*50)
    else:
        print("\nОшибка: Ни один файл не был успешно реконструирован. Проверьте структуру папок.")


### Сравнение RMSE

Теперь сравним полученный средний RMSE с предыдущим значением, чтобы оценить эффект от изменений.

In [ ]:
if 'all_rmse_values' in locals() and all_rmse_values:
    print(f"Предыдущий Mean RMSE: {np.mean(all_rmse_values):.5f} м.")

if 'all_rmse_values_v2' in locals() and all_rmse_values_v2:
    print(f"Новый Mean RMSE (v2): {np.mean(all_rmse_values_v2):.5f} м.")

    if 'all_rmse_values' in locals() and all_rmse_values:
        improvement = np.mean(all_rmse_values) - np.mean(all_rmse_values_v2)
        if improvement > 0:
            print(f"Улучшение RMSE: {improvement:.5f} м.")
        elif improvement < 0:
            print(f"Ухудшение RMSE: {-improvement:.5f} м.")
        else:
            print("RMSE остался без изменений.")


In [ ]:
# Упаковываем все сгенерированные файлы мешей в один архив
!zip -r reconstructed_models.zip /content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_valve

# Скачиваем архив на локальный диск компьютера
from google.colab import files
files.download('reconstructed_models.zip')


In [ ]:
import os
import glob
import traceback
import open3d as o3d
from plyfile import PlyData
import numpy as np # Убедимся, что numpy импортирован

# Укажите ваши точные пути
DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"
TARGET_FOLDER_NAME = "15"

target_path = os.path.join(DATASET_DIR, TARGET_FOLDER_NAME)
pcd_files = sorted(glob.glob(os.path.join(target_path, "*.[pP][lL][yY]")))

if not pcd_files:
    print(f"Ошибка: Скрипт всё ещё не видит файлы в {target_path}")
else:
    test_file = pcd_files[0]
    print(f"Запуск глубокой отладки для файла: {os.path.basename(test_file)}\n" + "="*50)

    try:
        # Шаг 1: Проверка чтения структуры PLY
        print("[Отладка 1/4] Чтение структуры через plyfile...")
        plydata = PlyData.read(test_file)
        vertex_data = plydata['vertex']
        available_properties = [p.name for p in vertex_data.properties]
        print(f"-> Успешно. Свойства в файле: {available_properties}")

        # Шаг 2: Вызов вашей функции загрузки (без скрытия ошибок)
        print("\n[Отладка 2/4] Запуск функции load_and_preprocess_pcd...")
        pcd, labels = load_and_preprocess_pcd(test_file)
        if pcd is None:
            print("-> Ошибка: Функция вернула None. Либо меток нет, либо точек < 10 после фильтрации шума.")
        else:
            print(f"-> Успешно. Точек после downsampling: {len(pcd.points)}, уникальных меток: {len(np.unique(labels))}")

            # Шаг 3: Тест сегментации и классификации
            print("\n[Отладка 3/4] Тест сегментации...")
            segments = split_into_segments(pcd, labels)
            print(f"-> Сформировано валидных сегментов: {len(segments)}")

            # Шаг 4: Тест реконструкции первого сегмента
            print("\n[Отладка 4/4] Тест триангуляции первого сегмента...")
            if segments:
                first_label = list(segments.keys())[0]
                seg_pcd = segments[first_label]
                geom_type, features = analyze_and_classify_segment(seg_pcd) # Получаем также признаки
                print(f"-> Класс первого сегмента: {geom_type}")
                print("   Вычисленные признаки:")
                for key, value in features.items():
                    print(f"     - {key}: {value:.4f}")

                mesh = reconstruct_segment(seg_pcd, geom_type)
                if mesh is None:
                    print("-> Ошибка: Реконструкция сегмента вернула None.")
                else:
                    print(f"-> Успешно! Создан меш с {len(mesh.triangles)} полигонами.")
                    # Print statements moved to B2kJLihWUqyr for visualization context
            else:
                print("-> Ошибка: Нет сегментов крупнее min_segment_points!")

    except Exception as e:
        print("\n!!! КРИТИЧЕСКАЯ ОШИБКА В ПАЙПЛАЙНЕ !!!")
        traceback.print_exc()

Запуск глубокой отладки для файла: valve_0001_lidar_classes.ply
[Отладка 1/4] Чтение структуры через plyfile...
-> Успешно. Свойства в файле: ['x', 'y', 'z', 'scalar_Label']

[Отладка 2/4] Запуск функции load_and_preprocess_pcd...
-> Успешно. Точек после downsampling: 10060, уникальных меток: 8

[Отладка 3/4] Тест сегментации...
-> Сформировано валидных сегментов: 8

[Отладка 4/4] Тест триангуляции первого сегмента...
-> Класс первого сегмента: трубчатый объект
Исходные точки: [[ 124.67668152   69.97530365 -175.39720154]
 [ 124.67576599   76.85115814 -174.23048401]
 [ 124.68023682   56.12125397 -172.83676147]]
Вершины меша: [[305.57177734 132.38180542 -20.30316162]
 [306.25408936 132.69158936 -20.30316162]
 [308.48236084 134.99795532 -20.30316162]]
-> Успешно! Создан меш с 4524 полигонами.


In [ ]:
# Эта ячейка теперь избыточна, так как ее содержимое было перемещено в B2kJLihWUqyr для лучшего контекста и интерактивной визуализации.

NameError: name 'source_pcd' is not defined